# DX702 Quiz 3

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf

random_state    = 42
testsize        = 0.2


For questions 1 and 2: 

Given a dataset with time series data containing an event, use a linear regression to test whether there was a `discontinuity` in the data at the event.

 Consider the possibility, first, of a discontinuity only in the value of the variable but not the derivative. Then consider that there may be a discontinuity in the first derivative (the slope).  

Use the file `homework_3.1.csv`. 

In [2]:
data = pd.read_csv('homework_3.1.csv')
data = data.drop(columns=['Unnamed: 0'])

In [3]:
data.head(n=10)

,time,value1,value2,value3
0,0,1.764052,1.883151,-0.369182
1,1,0.420157,-1.327759,-0.219379
2,2,1.018738,-1.230485,1.139660
3,3,2.300893,1.029397,0.715264
4,4,1.947558,-1.093123,0.720132
5,5,-0.877278,2.043621,-1.516956
6,6,1.070088,-0.293619,0.095674
7,7,-0.011357,-0.607455,-0.598031
8,8,0.056781,2.082942,0.439925
9,9,0.590599,1.660515,0.081850


In [4]:
data.describe()

,time,value1,value2,value3
count,100.000000,100.000000,100.000000,100.000000
mean,49.500000,2.152308,1.807013,2.155768
std,29.011492,2.052524,1.557664,2.126580
min,0.000000,-2.152990,-1.327759,-1.516956
25%,24.750000,0.543743,0.963499,0.340994
50%,49.500000,1.948924,1.723450,1.816849
75%,74.250000,3.433771,2.796829,3.912036
max,99.000000,7.075870,6.083236,6.983917


## Question 1

Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the value at the event (at time = 50)? 

Option A
value1 

Option B
value2

`Option C
value3` 

In [5]:
# Create the event_dummy variable for discontinuity in VALUE
# This variable is 0 for time < 50 and 1 for time >= 50
data['event_dummy'] = (data['time'] >= 50).astype(int)

# Create the INTERACTION TERM for discontinuity in SLOPE
# This captures the change in slope after the event
data['time_after_event'] = data['time'] * data['event_dummy']

value_columns = ['value1', 'value2', 'value3']

print("--- Analysis for Discontinuity in Value (Intercept Jump) ---")
# Dictionary to store regression results (p-value and coefficient for event_dummy) for value discontinuity
discontinuity_value_results = {}

# Perform regression for each value column to check for discontinuity in value
for col in value_columns:
    # Model: value ~ time + event_dummy
    # 'time' accounts for the pre-existing trend, 'event_dummy' captures the jump in value at time=50
    formula = f'{col} ~ time + event_dummy'
    model   = smf.ols(formula=formula, data=data)
    results = model.fit()

    # Get the p-value and coefficient for event_dummy
    p_value     = results.pvalues['event_dummy']
    coefficient = results.params['event_dummy']

    discontinuity_value_results[col] = {'p_value': p_value, 'coefficient': coefficient}

    print(f"Dataset: {col}")
    print(f"  Event Dummy Coefficient: {coefficient:.4f}")
    print(f"  P-value: {p_value:.4f}")
    print("-" * 30)


--- Analysis for Discontinuity in Value (Intercept Jump) ---
Dataset: value1
  Event Dummy Coefficient: 0.8508
  P-value: 0.0856
------------------------------
Dataset: value2
  Event Dummy Coefficient: 0.6827
  P-value: 0.1131
------------------------------
Dataset: value3
  Event Dummy Coefficient: 1.7673
  P-value: 0.0000
------------------------------


## Question 2

Which dataset is least likely to have a discontinuity (or has the strongest discontinuity) in the derivative (at time = 50)? 

Option A
value2

`Option B
value1`

Option C
value3


In [6]:
# Dictionary to store regression results for discontinuity in slope
discontinuity_slope_results = {}

# Perform regression for each value column to check for discontinuity in slope
for col in value_columns:
    # Model: value ~ time + event_dummy + time_after_event
    # 'time' accounts for the pre-event slope.
    # 'event_dummy' accounts for a jump in value at the event (intercept discontinuity).
    # 'time_after_event' accounts for a change in slope after the event (derivative discontinuity).
    formula     = f'{col} ~ time + event_dummy + time_after_event'
    model       = smf.ols(formula=formula, data=data)
    results     = model.fit()

    # Get the p-value and coefficient for the interaction term (time_after_event)
    p_value     = results.pvalues['time_after_event']
    coefficient = results.params['time_after_event']

    discontinuity_slope_results[col] = {'p_value': p_value, 'coefficient': coefficient}

    print(f"Dataset: {col}")
    print(f"  Time After Event (Slope Change) Coefficient: {coefficient:.4f}")
    print(f"  P-value: {p_value:.4f}")
    print("-" * 30)

Dataset: value1
  Time After Event (Slope Change) Coefficient: 0.1053
  P-value: 0.0000
------------------------------
Dataset: value2
  Time After Event (Slope Change) Coefficient: 0.0369
  P-value: 0.0118
------------------------------
Dataset: value3
  Time After Event (Slope Change) Coefficient: 0.0507
  P-value: 0.0002
------------------------------


For questions 3 to 5:  

Given a dataset with treatment and control data having “before” and “after” parts, apply a `differences-in-differences` regression.  
Use `homework_3.2.a.csv` and `homework_3.2.b.csv`. 

In [7]:
df_a = pd.read_csv('homework_3.2.a.csv')
df_a = df_a.drop(columns=['Unnamed: 0'])

In [8]:
df_a.head(n=10)

,group1,time1,outcome1
0,0,0,0.882026
1,0,1,1.600079
2,0,0,0.489369
3,0,1,2.520447
4,0,0,0.933779
5,0,1,0.911361
6,0,0,0.475044
7,0,1,1.324321
8,0,0,-0.051609
9,0,1,1.605299


In [9]:
df_b = pd.read_csv('homework_3.2.b.csv')
df_b = df_b.drop(columns=['Unnamed: 0'])

In [10]:
df_b.head(n=10)

,group2,time2,outcome2
0,0,0,0.667155
1,0,1,2.470969
2,0,0,-0.506778
3,0,1,1.525657
4,0,0,0.273664
5,0,1,1.641776
6,0,0,0.648928
7,0,1,-0.781693
8,0,0,-0.059189
9,0,1,1.686840


In [11]:

# Differences-in-Differences regression for df_a
# outcome1 ~ C(group1) + C(time1) + C(group1):C(time1)
# C() ensures that the variables are treated as categorical
model_a = smf.ols('outcome1 ~ C(group1) * C(time1)', data=df_a).fit()
print("Regression results for df_a:")
print(model_a.summary().as_text())

# Differences-in-Differences regression for df_b
# outcome2 ~ C(group2) + C(time2) + C(group2):C(time2)
model_b = smf.ols('outcome2 ~ C(group2) * C(time2)', data=df_b).fit()
print("\nRegression results for df_b:")
print(model_b.summary().as_text())


Regression results for df_a:
                            OLS Regression Results                            
Dep. Variable:               outcome1   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.899
Method:                 Least Squares   F-statistic:                     2964.
Date:                Mon, 09 Jun 2025   Prob (F-statistic):               0.00
Time:                        18:41:50   Log-Likelihood:                -712.28
No. Observations:                1000   AIC:                             1433.
Df Residuals:                     996   BIC:                             1452.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------

In [12]:

# Extracting the treatment effects and p-values
# For model_a, the interaction term is C(group1)[T.1]:C(time1)[T.1]
treatment_effect_a = model_a.params['C(group1)[T.1]:C(time1)[T.1]']
p_value_a = model_a.pvalues['C(group1)[T.1]:C(time1)[T.1]']

# For model_b, the interaction term is C(group2)[T.1]:C(time2)[T.1]
treatment_effect_b = model_b.params['C(group2)[T.1]:C(time2)[T.1]']
p_value_b = model_b.pvalues['C(group2)[T.1]:C(time2)[T.1]']

print(f"\nTreatment Effect, Group 1 (df_a): {treatment_effect_a:.4f} (p-value: {p_value_a:.8f})")
print(f"Treatment Effect, Group 2 (df_b): {treatment_effect_b:.4f} (p-value: {p_value_b:.8f})")




Treatment Effect, Group 1 (df_a): 0.6858 (p-value: 0.00000000)
Treatment Effect, Group 2 (df_b): 1.3499 (p-value: 0.00000000)


## Question 3

Which dataset likely has the `largest treatment effect`, assuming that the treatment and control groups have parallel trends? 

Option A
Group 1

Option B
Group 2



In [13]:
if abs(treatment_effect_a) > abs(treatment_effect_b):
    q3_answer = "Group 1"
else:
    q3_answer = "Group 2"
print(f'Answer: {q3_answer}')

Answer: Group 2


## Question 4

Using the standard errors for regression, which dataset has the most statistically significant (and nonzero) treatment effect? 

Option A
Group 1

Option B
Group 2


In [14]:
if p_value_a < p_value_b:
    q4_answer = "Group 1"
else:
    q4_answer = "Group 2"
print(f'ANSWER: {q4_answer}')

ANSWER: Group 1


## Question 5

Which of these is closest to the treatment effect for group 2? 

Option A
0.5831

Option B
1.248 

Option C
4.209


In [15]:
options_q5 = {
    "A": 0.5831,
    "B": 1.248,
    "C": 4.209
}
closest_option_q5 = min(options_q5, key=lambda x: abs(options_q5[x] - treatment_effect_b))
print(f'ANSWER: {closest_option_q5}')

ANSWER: B
